In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupKFold, cross_val_score
from tqdm.notebook import tqdm
import joblib
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold 


In [ ]:
from sklearn.preprocessing import LabelEncoder
# x_train
x_train = ### All the variables you want to use for prediction

y_train = ### whatever your prediction variable is 

le = LabelEncoder()
y_train = le.fit_transform(y_train)

num_classes = len(np.unique(y_train))
print(f"Dataset shape: {x_train.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")


In [ ]:
from datetime import datetime

def timer(start_time=None):
    if not start_time:
        start_time = datetime.now()
        return start_time
    elif start_time:
        thour, temp_sec = divmod((datetime.now() - start_time).total_seconds(), 3600)
        tmin, tsec = divmod(temp_sec, 60)
        print('\n Time taken: %i hours %i minutes and %s seconds.' % (thour, tmin, round(tsec, 2)))

In [ ]:
gss = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(x_train, y_train, groups=groups))

train_groups = groups[train_idx]
test_groups = groups[test_idx]

overlap = np.intersect1d(train_groups, test_groups)
print("overlap count:", len(overlap))
print("example overlaps:", overlap[:10])

X_train_full, X_test = x_train.iloc[train_idx], x_train.iloc[test_idx]
y_train_full, y_test = y_train[train_idx], y_train[test_idx]
groups_train_full = groups[train_idx]

In [ ]:
assert len(x_train) == len(y_train) == len(groups)
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))

In [ ]:
from skopt.space import Real, Integer
from skopt import BayesSearchCV
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

param_space = {
    # Tree complexity (tighter)
    "max_depth": Integer(2, 6),
    "min_child_weight": Integer(3, 20),
    "gamma": Real(0.0, 5.0, prior="uniform"),

    # Row/feature subsampling
    "subsample": Real(0.5, 0.9, prior="uniform"),
    "colsample_bytree": Real(0.4, 0.9, prior="uniform"),
    "colsample_bylevel": Real(0.4, 1.0, prior="uniform"),
    "scale_pos_weight": Real(0.4, 1.0, prior="uniform"),

    # Learning rate
    "learning_rate": Real(0.01, 0.1, prior="log-uniform"),

    # Regularization (critical)
    "reg_lambda": Real(1.0, 30.0, prior="log-uniform"),
    "reg_alpha": Real(1e-4, 10.0, prior="log-uniform"),
}

# BUILDS A DICTIONARY TO PLACE OUR CV RESULTS FOR THE ROC CURVE
metrics = ['auc', 'fpr', 'tpr', 'thresholds']
results = {
    'train': {m:[] for m in metrics},
    'val'  : {m:[] for m in metrics},
    'test' : {m:[] for m in metrics}
}

In [ ]:
import warnings
from sklearn.exceptions import FitFailedWarning
from sklearn.metrics import roc_curve, roc_auc_score

# Set model
sklearn_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric="auc",
    scale_pos_weight = 2.48,
    enable_categorical = True,
    n_jobs=-1,
    random_state=42
)
######### Hyperparameter Search ##########
warnings.filterwarnings("ignore", category=FitFailedWarning)

inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# ---------- Non-nested ----------
clf_non_nested = BayesSearchCV(estimator=sklearn_model, search_spaces=param_space, n_iter=5, cv=outer_cv, scoring="roc_auc", 
                 refit=True, n_jobs=-1, verbose=1, random_state = 42)

with joblib.parallel_backend('threading'):
    clf_non_nested.fit(X_train_full, y_train_full, groups = groups_train_full)

non_nested_scores = clf_non_nested.best_score_

print("Non-nested test score:", non_nested_scores)

# ---------- Nested ----------
clf_nested = BayesSearchCV(estimator=sklearn_model, search_spaces=param_space, n_iter=5, cv=inner_cv, scoring="roc_auc", 
             refit=True, n_jobs=-1, verbose=1, random_state = 42)

val_aucs = []

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X_train_full, y_train_full, groups_train_full)
):
    print(f"Outer fold {fold+1}")
    
    X_train = X_train_full.iloc[train_idx]
    X_test  = X_train_full.iloc[test_idx]
    y_train = y_train_full[train_idx]
    y_test  = y_train_full[test_idx]
    groups_train = groups_train_full[train_idx]

    # Fit inner CV (Bayesian search)
    clf_nested.fit(X_train, y_train, groups=groups_train)
    best_model = clf_nested.best_estimator_

    # ---- TRAIN ----
    y_train_proba = best_model.predict_proba(X_train)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_train, y_train_proba)
    results['train']['fpr'].append(fpr)
    results['train']['tpr'].append(tpr)
    results['train']['thresholds'].append(thresholds)
    results['train']['auc'].append(roc_auc_score(y_train, y_train_proba))

    # ---- VALIDATION ----
    y_test_proba = best_model.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
    auc_score = roc_auc_score(y_test, y_test_proba)

    results['val']['fpr'].append(fpr)
    results['val']['tpr'].append(tpr)
    results['val']['thresholds'].append(thresholds)
    results['val']['auc'].append(auc_score)

    val_aucs.append(auc_score)

print(f"\nNested ROC AUC: {np.mean(val_aucs):.4f} ± {np.std(val_aucs):.4f}")
nested_scores = np.array(val_aucs)

# ---------- Comparison ----------
score_difference = non_nested_scores - nested_scores

print(
    "Average difference of {:6f} with std. dev. of {:6f}.".format(
        score_difference.mean(),
        score_difference.std()
    )
)

#########################################

In [ ]:
#########################################
b_results = pd.DataFrame([clf_non_nested.best_params_])
b_results.to_csv('save_your_model_params.csv', index=False)

#########################################
# This step isn't entirely necessary
final_model = clf_non_nested.best_estimator_
y_pred = final_model.predict(X_test)
best_params = clf_non_nested.best_params_

joblib.dump(final_model, 'grouped_pN_bayesian_search_model.joblib')

#########################################

In [ ]:
# 2. FINAL TRAIN/VALIDATION SPLIT FOR MONITORING
gss = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X_train_full, y_train_full, groups=groups_train_full))

X_train = X_train_full.iloc[train_idx]
X_val = X_train_full.iloc[val_idx]
y_train = y_train_full[train_idx]
y_val = y_train_full[val_idx]
groups_train = groups_train_full[train_idx]
groups_val = groups_train_full[val_idx]

def class_distribution(y):
    return pd.Series(y).value_counts(normalize=True)

print("Original:")
print(class_distribution(y))

print("Full training set:")
print(class_distribution(y_train_full))

print("\nTrain:")
print(class_distribution(y_train))

print("\nValidation:")
print(class_distribution(y_val))

# Convert to DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)

In [ ]:
# 3. TRAINING WITH PROGRESS MONITORING + EARLY STOPPING + HYPERPARAMETER PARAMS

params = {
    "objective": "multi:softprob",
    "num_class": num_classes,
    "eval_metric": "mlogloss",
    "eta": best_params["learning_rate"],
    "max_depth": best_params["max_depth"],
    "seed": 42
}
evals = [(dtrain, "train"), (dval, "validation")]
evals_result = {}

bst = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=2000,
    evals=evals,
    evals_result=evals_result,
    early_stopping_rounds=50,
    verbose_eval=10
)

bst.save_model("xgboost_pN_hyperparameter.model")

print(f"\nBest iteration: {bst.best_iteration}")
####################################

In [ ]:
import plotly.graph_objects as go

kind = 'val'
c_fill      = 'rgba(52, 152, 219, 0.2)'
c_line      = 'rgba(52, 152, 219, 0.5)'
c_line_main = 'rgba(41, 128, 185, 1.0)'
c_grid      = 'rgba(189, 195, 199, 0.5)'
c_annot     = 'rgba(149, 165, 166, 0.5)'
c_highlight = 'rgba(192, 57, 43, 1.0)'
fpr_mean    = np.linspace(0, 1, 100)
interp_tprs = []
for i in range(5):
    fpr           = results[kind]['fpr'][i]
    tpr           = results[kind]['tpr'][i]
    interp_tpr    = np.interp(fpr_mean, fpr, tpr)
    interp_tpr[0] = 0.0
    interp_tprs.append(interp_tpr)
tpr_mean     = np.mean(interp_tprs, axis=0)
tpr_mean[-1] = 1.0
tpr_std      = 2*np.std(interp_tprs, axis=0)
tpr_upper    = np.clip(tpr_mean+tpr_std, 0, 1)
tpr_lower    = tpr_mean-tpr_std
auc          = np.mean(results[kind]['auc'])
fig = go.Figure([
    go.Scatter(
        x          = fpr_mean,
        y          = tpr_upper,
        line       = dict(color=c_line, width=1),
        hoverinfo  = "skip",
        showlegend = False,
        name       = 'upper'),
    go.Scatter(
        x          = fpr_mean,
        y          = tpr_lower,
        fill       = 'tonexty',
        fillcolor  = c_fill,
        line       = dict(color=c_line, width=1),
        hoverinfo  = "skip",
        showlegend = False,
        name       = 'lower'),
    go.Scatter(
        x          = fpr_mean,
        y          = tpr_mean,
        line       = dict(color=c_line_main, width=2),
        hoverinfo  = "skip",
        showlegend = True,
        name       = f'AUC: {auc:.3f}')
])
fig.add_shape(
    type ='line', 
    line =dict(dash='dash'),
    x0=0, x1=1, y0=0, y1=1
)
fig.update_layout(
    template    = 'plotly_white', 
    title_x     = 0.5,
    xaxis_title = "1 - Specificity",
    yaxis_title = "Sensitivity",
    width       = 800,
    height      = 800,
    legend      = dict(
        yanchor="bottom", 
        xanchor="right", 
        x=0.95,
        y=0.01,
    )
)
fig.update_yaxes(
    range       = [0, 1],
    gridcolor   = c_grid,
    scaleanchor = "x", 
    scaleratio  = 1,
    linecolor   = 'black')
fig.update_xaxes(
    range       = [0, 1],
    gridcolor   = c_grid,
    constrain   = 'domain',
    linecolor   = 'black')

In [ ]:
def normalize_confusion_matrix(cm, norm='true'):
    """
    Normalize a confusion matrix.
    
    Parameters:
    cm (array-like): Confusion matrix to be normalized.
    norm (str): Type of normalization ('true', 'pred', 'all').
    
    Returns:
    ndarray: Normalized confusion matrix.
    """
    if norm == 'true':
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    elif norm == 'pred':
        cm_normalized = cm.astype('float') / cm.sum(axis=0)[np.newaxis, :]
    elif norm == 'all':
        cm_normalized = cm.astype('float') / cm.sum()
    else:
        raise ValueError("Unknown normalization type. Use 'true', 'pred', or 'all'.")
    
    return cm_normalized

In [ ]:
bst = xgb.Booster()
bst.load_model("your_model.model")

In [ ]:
from sklearn import preprocessing as p
from sklearn.metrics import confusion_matrix

# 5. PREDICTIONS AND EVALUATION
y_pred_train = bst.predict(dtrain)[:, 1]
y_pred_train_r = [round(value) for value in y_pred_train]
y_pred_val = bst.predict(dval)[:, 1]
y_pred_val_r = [round(value) for value in y_pred_val]
y_pred_test = bst.predict(dtest)[:, 1]
y_pred_test_r = [round(value) for value in y_pred_test]

print("\n=== FINAL EVALUATION ===")
print("Train accuracy:", accuracy_score(y_train, y_pred_train_r))
print("Validation accuracy:", accuracy_score(y_val, y_pred_val_r))
print("Test accuracy:", accuracy_score(y_test, y_pred_test_r))

print("\nTest classification report:")
print(classification_report(y_test, y_pred_test_r))

categories = df['pN'].cat.categories
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test_r)
cm_normalized = normalize_confusion_matrix(cm, norm='true')

labels = np.empty_like(cm).astype(str)
for i in range(cm.shape[0]):
    for k in range(cm_normalized.shape[1]):
        count = cm[i,k]
        percent = cm_normalized[i,k]
        labels[i,k] = f"{percent:.0%}\n({count})"
        

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=labels,
    fmt="",              # show decimals
    cmap="Blues",
    xticklabels=categories,
    yticklabels=categories,
    cbar=True
)

plt.title("Confusion Matrix", fontsize=16)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)

plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()
# Feature importance
xgb.plot_importance(bst, max_num_features=50)
plt.show()

In [ ]:
# 4. PLOT TRAINING PROGRESS
train_loss = evals_result["train"]["mlogloss"]
val_loss = evals_result["validation"]["mlogloss"]
epochs = range(1, len(train_loss) + 1)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, label="Train mlogloss", linewidth=2)
plt.plot(epochs, val_loss, label="Validation mlogloss", linewidth=2)
plt.axvline(bst.best_iteration, color='red', linestyle='--', label=f'Best iter: {bst.best_iteration}')
plt.xlabel("Boosting rounds")
plt.ylabel("Multi-class log loss")
plt.title("XGBoost Training Progress")
plt.legend()
plt.grid(True, alpha=0.3)


In [ ]:
from sklearn import preprocessing as p
from sklearn.metrics import confusion_matrix

# 5. PREDICTIONS AND EVALUATION
y_pred_train = bst.predict(dtrain).argmax(axis=1)
y_pred_val = bst.predict(dval).argmax(axis=1)
y_pred_test = bst.predict(dtest).argmax(axis=1)

print("\n=== FINAL EVALUATION ===")
print("Train accuracy:", accuracy_score(y_train, y_pred_train))
print("Validation accuracy:", accuracy_score(y_val, y_pred_val))
print("Test accuracy:", accuracy_score(y_test, y_pred_test))

print("\nTest classification report:")
print(classification_report(y_test, y_pred_test))

cm = confusion_matrix(y_test, y_pred_test)
print(cm)
cm_normalized = normalize_confusion_matrix(cm, norm='true')


plt.figure(figsize=(4, 4))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",              # show decimals
    cmap="Blues",
    xticklabels=categories,
    yticklabels=categories,
    cbar=True
)

plt.title("Confusion Matrix", fontsize=16)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)

plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

# Feature importance
xgb.plot_importance(bst, max_num_features=50)
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

auc_score = roc_auc_score(y_test, y_pred_test)
print(f"ROC-AUC Score on Test Set: {auc_score:.4f}")